In [1]:
# ============================================================
# M1 ACCOUNT A — RECORD 114 LEAD ORIENTATION CHECK
# CELL 1: SETUP AND IMPORTS
# ============================================================

import os
import re
import hashlib
import unicodedata

import numpy as np
import pandas as pd

from collections import Counter

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

print("=" * 65)
print("M1 ACCOUNT A — RECORD 114 LEAD ORIENTATION CHECK")
print("=" * 65)

print("Python diagnostic environment ready.")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\nNo dataset has been modified.")
print("No model has been trained.")

M1 ACCOUNT A — RECORD 114 LEAD ORIENTATION CHECK
Python diagnostic environment ready.
NumPy: 2.1.3
Pandas: 2.3.3

No dataset has been modified.
No model has been trained.


In [2]:
# ============================================================
# CELL 2: AUTOMATIC INPUT FILE DISCOVERY
# ============================================================

INPUT_ROOT = "/kaggle/input"

def clean_column_name(name):
    name = unicodedata.normalize("NFKC", str(name))
    return name.strip().strip('"').strip("'").strip()


def normalise_name(name):
    name = clean_column_name(name).casefold()
    return re.sub(r"[^a-z0-9#]", "", name)


# Expected waveform columns
MLII_COLUMNS = [f"MLII_{i}" for i in range(201)]
LEAD2_COLUMNS = [f"Lead2_{i}" for i in range(201)]

WAVEFORM_COLUMNS = MLII_COLUMNS + LEAD2_COLUMNS

assert len(WAVEFORM_COLUMNS) == 402


# File candidate lists
processed_candidates = []
signal_candidates = []
annotation_candidates = []


# ------------------------------------------------------------
# Search Kaggle input folders
# ------------------------------------------------------------

for root, dirs, files in os.walk(INPUT_ROOT):

    for filename in files:

        full_path = os.path.join(root, filename)

        lower_name = filename.lower()
        lower_path = full_path.lower()

        # Original record 114 ECG signal
        if lower_name == "114.csv" and "signals" in lower_path:
            signal_candidates.append(full_path)

        # Original record 114 annotation
        annotation_stem = os.path.splitext(lower_name)[0]

        if (
            "annotation" in lower_path
            and re.match(
                r"^114(?:$|[^0-9])",
                annotation_stem
            )
        ):
            annotation_candidates.append(full_path)

        # Identify processed dataset using its columns
        if lower_name.endswith(".csv"):

            try:
                header_df = pd.read_csv(full_path, nrows=0)

                headers = {
                    clean_column_name(c)
                    for c in header_df.columns
                }

                required = {
                    "Record_ID",
                    "Label",
                    "MLII_0",
                    "MLII_200",
                    "Lead2_0",
                    "Lead2_200"
                }

                if required.issubset(headers):
                    processed_candidates.append(full_path)

            except Exception:
                continue


# ------------------------------------------------------------
# Display discovered candidates
# ------------------------------------------------------------

print("PROCESSED DATASET CANDIDATES:")
for p in processed_candidates:
    print(p)

print("\nRECORD 114 SIGNAL FILES:")
for p in signal_candidates:
    print(p)

print("\nRECORD 114 ANNOTATION FILES:")
for p in annotation_candidates:
    print(p)


# ------------------------------------------------------------
# Select processed dataset
# ------------------------------------------------------------

if not processed_candidates:
    raise FileNotFoundError(
        "Processed heartbeat dataset was not found."
    )

if len(processed_candidates) == 1:
    processed_path = processed_candidates[0]

else:
    preferred = [
        p for p in processed_candidates
        if "validated_aami_dataset" in os.path.basename(p).lower()
    ]

    if len(preferred) == 1:
        processed_path = preferred[0]

    else:
        processed_path = max(
            processed_candidates,
            key=os.path.getsize
        )


# ------------------------------------------------------------
# Select original signal file
# ------------------------------------------------------------

if len(signal_candidates) != 1:
    raise RuntimeError(
        f"Expected one original 114.csv file, "
        f"found {len(signal_candidates)}. "
        f"Candidates: {signal_candidates}"
    )

signal_path = signal_candidates[0]


# ------------------------------------------------------------
# Select original annotation file
# ------------------------------------------------------------

if len(annotation_candidates) != 1:
    raise RuntimeError(
        f"Expected one record 114 annotation file, "
        f"found {len(annotation_candidates)}. "
        f"Candidates: {annotation_candidates}"
    )

annotation_path = annotation_candidates[0]


# ------------------------------------------------------------
# Display final selections
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SELECTED FILES")
print("=" * 65)

print("Processed dataset:", processed_path)
print("Original ECG signal:", signal_path)
print("Original annotations:", annotation_path)

print("\nCELL 2 COMPLETE.")

PROCESSED DATASET CANDIDATES:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv

RECORD 114 SIGNAL FILES:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Signals/114.csv

RECORD 114 ANNOTATION FILES:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/114annotations.txt

SELECTED FILES
Processed dataset: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv
Original ECG signal: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Signals/114.csv
Original annotations: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/02_Original_Recordings/Annotations/114annotations.txt

CELL 2 COMPLETE.


In [3]:
# ============================================================
# CELL 3: LOAD AND INSPECT ORIGINAL AND PROCESSED FILES
# ============================================================

# ------------------------------------------------------------
# 1. Load processed heartbeat dataset
# ------------------------------------------------------------

processed_df = pd.read_csv(processed_path)

processed_df.columns = [
    clean_column_name(c)
    for c in processed_df.columns
]

processed_df["Record_ID"] = pd.to_numeric(
    processed_df["Record_ID"],
    errors="raise"
).astype(int)

processed_df["Label"] = (
    processed_df["Label"]
    .astype(str)
    .str.strip()
)

print("PROCESSED DATASET")
print("Shape:", processed_df.shape)
print("Record count:", processed_df["Record_ID"].nunique())

missing_columns = [
    c for c in WAVEFORM_COLUMNS
    if c not in processed_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing waveform columns: {missing_columns}"
    )

print("402 waveform columns: PASS")


# ------------------------------------------------------------
# 2. Load original signal file
# ------------------------------------------------------------

signal_df = pd.read_csv(signal_path)

signal_df.columns = [
    clean_column_name(c)
    for c in signal_df.columns
]


def find_column(df, expected_name):

    matches = [
        c for c in df.columns
        if normalise_name(c) == expected_name
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Could not uniquely identify {expected_name}. "
            f"Matches: {matches}"
        )

    return matches[0]


sample_col = find_column(signal_df, "sample#")
mlii_col = find_column(signal_df, "mlii")

other_leads = [
    c for c in signal_df.columns
    if c not in [sample_col, mlii_col]
]

if len(other_leads) != 1:
    raise ValueError(
        f"Expected one second lead. Found: {other_leads}"
    )

second_lead_col = other_leads[0]


# Convert signal columns to numeric
signal_df[sample_col] = pd.to_numeric(
    signal_df[sample_col],
    errors="raise"
).astype(int)

signal_df[mlii_col] = pd.to_numeric(
    signal_df[mlii_col],
    errors="raise"
)

signal_df[second_lead_col] = pd.to_numeric(
    signal_df[second_lead_col],
    errors="raise"
)


print("\nORIGINAL SIGNAL FILE")
print("Shape:", signal_df.shape)
print("Sample column:", sample_col)
print("MLII column:", mlii_col)
print("Second lead:", second_lead_col)

display(
    signal_df[
        [sample_col, mlii_col, second_lead_col]
    ].head(10)
)


# ------------------------------------------------------------
# 3. Display annotation file preview
# ------------------------------------------------------------

print("\nORIGINAL ANNOTATION FILE — FIRST 10 LINES")

with open(
    annotation_path,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for i, line in enumerate(f):

        if i == 10:
            break

        print(repr(line.rstrip("\n")))


print("\nCELL 3 COMPLETE.")

PROCESSED DATASET
Shape: (106870, 404)
Record count: 46
402 waveform columns: PASS

ORIGINAL SIGNAL FILE
Shape: (650000, 3)
Sample column: sample #
MLII column: MLII
Second lead: V5


,sample #,MLII,V5
0,0,1027,1015
1,1,1027,1015
2,2,1027,1015
3,3,1027,1015
4,4,1027,1015
5,5,1027,1015
6,6,1027,1015
7,7,1027,1015
8,8,1027,1015
9,9,1032,1015



ORIGINAL ANNOTATION FILE — FIRST 10 LINES
'      Time   Sample #  Type  Sub Chan  Num\tAux'
'    0:00.058       21     +    0    0    0\t(N'
'    0:00.861      310     N    0    0    0'
'    0:02.017      726     N    0    0    0'
'    0:03.175     1143     N    0    0    0'
'    0:04.314     1553     N    0    0    0'
'    0:05.458     1965     N    0    0    0'
'    0:06.550     2358     N    0    0    0'
'    0:07.617     2742     N    0    0    0'
'    0:08.794     3166     N    0    0    0'

CELL 3 COMPLETE.


In [4]:
# ============================================================
# CELL 4: RECONSTRUCT RECORD 114 FROM ORIGINAL FILES
# ============================================================

# ------------------------------------------------------------
# 1. Define AAMI annotation mapping
# ------------------------------------------------------------

AAMI_MAP = {
    "N": "N",
    "L": "N",
    "R": "N",
    "e": "N",
    "j": "N",

    "A": "S",
    "a": "S",
    "J": "S",
    "S": "S",

    "V": "V",
    "E": "V",

    "F": "F",

    "/": "Q",
    "f": "Q",
    "Q": "Q"
}

AAMI_CLASSES = ["N", "S", "V", "F", "Q"]


# ------------------------------------------------------------
# 2. Parse original annotation file
# ------------------------------------------------------------

annotation_rows = []

parser_errors = []

with open(
    annotation_path,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for line_number, line in enumerate(f, start=1):

        line = line.strip()

        if not line:
            continue

        parts = re.split(r"\s+", line, maxsplit=6)

        if len(parts) < 3:
            continue

        try:
            sample_index = int(parts[1])

        except ValueError:
            # Header and non-data lines are ignored.
            continue

        original_symbol = parts[2]

        # Keep only valid AAMI beat symbols.
        # Rhythm markers such as + are excluded.
        if original_symbol in AAMI_MAP:

            annotation_rows.append({
                "Sample_Index": sample_index,
                "Original_Symbol": original_symbol,
                "AAMI_Label": AAMI_MAP[original_symbol]
            })


annotation_df = pd.DataFrame(annotation_rows)

if annotation_df.empty:
    raise ValueError(
        "No valid AAMI beat annotations were parsed."
    )


print("VALID ANNOTATION CANDIDATES:", len(annotation_df))

print("\nOriginal annotation symbol counts:")

display(
    annotation_df["Original_Symbol"]
    .value_counts()
    .to_frame("Count")
)


# ------------------------------------------------------------
# 3. Create sample-position lookup
# ------------------------------------------------------------

if signal_df[sample_col].duplicated().any():
    raise ValueError(
        "Duplicate sample positions found in original signal."
    )

sample_to_position = {
    int(sample): position
    for position, sample in enumerate(
        signal_df[sample_col].to_numpy()
    )
}


# ------------------------------------------------------------
# 4. Reconstruct all valid beat windows
# ------------------------------------------------------------

rebuilt_rows = []

boundary_skips = 0
missing_sample_windows = 0

for beat in annotation_df.itertuples(index=False):

    centre = int(beat.Sample_Index)

    # 100 samples before, centre, and 100 samples after.
    required_samples = range(
        centre - 100,
        centre + 101
    )

    positions = [
        sample_to_position.get(s)
        for s in required_samples
    ]

    if any(p is None for p in positions):

        if (
            centre - 100 < signal_df[sample_col].min()
            or centre + 100 > signal_df[sample_col].max()
        ):
            boundary_skips += 1

        else:
            missing_sample_windows += 1

        continue


    window = signal_df.iloc[positions]

    mlii_values = window[mlii_col].to_numpy(
        dtype=np.float64
    )

    second_values = window[second_lead_col].to_numpy(
        dtype=np.float64
    )

    if len(mlii_values) != 201 or len(second_values) != 201:
        raise ValueError(
            "Incorrect ECG window length."
        )


    row = {
        "Record_ID": 114,
        "Sample_Index": centre,
        "Original_Symbol": beat.Original_Symbol,
        "AAMI_Label": beat.AAMI_Label
    }

    # MLII values
    for i, value in enumerate(mlii_values):
        row[f"MLII_{i}"] = value

    # Second-lead values
    for i, value in enumerate(second_values):
        row[f"Lead2_{i}"] = value

    rebuilt_rows.append(row)


rebuilt_114 = pd.DataFrame(rebuilt_rows)


# ------------------------------------------------------------
# 5. Display reconstruction results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RECORD 114 RECONSTRUCTION SUMMARY")
print("=" * 65)

print("Reconstructed beats:", len(rebuilt_114))
print("Boundary windows skipped:", boundary_skips)
print("Other incomplete windows:", missing_sample_windows)

print("\nAAMI class counts:")

display(
    rebuilt_114["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
    .to_frame("Count")
)

print("\nFirst five reconstructed beats:")

display(
    rebuilt_114[
        [
            "Record_ID",
            "Sample_Index",
            "Original_Symbol",
            "AAMI_Label"
        ]
    ].head()
)

print("\nCELL 4 COMPLETE.")

VALID ANNOTATION CANDIDATES: 1879

Original annotation symbol counts:


,Count
Original_Symbol,
N,1820
V,43
A,10
F,4
J,2



RECORD 114 RECONSTRUCTION SUMMARY
Reconstructed beats: 1879
Boundary windows skipped: 0
Other incomplete windows: 0

AAMI class counts:


,Count
AAMI_Label,
N,1820
S,12
V,43
F,4
Q,0



First five reconstructed beats:


,Record_ID,Sample_Index,Original_Symbol,AAMI_Label
0,114,310,N,N
1,114,726,N,N
2,114,1143,N,N
3,114,1553,N,N
4,114,1965,N,N



CELL 4 COMPLETE.


In [5]:
# ============================================================
# CELL 5: RECORD 114 LEAD ORIENTATION VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Select record 114 from processed dataset
# ------------------------------------------------------------

processed_114 = processed_df[
    processed_df["Record_ID"] == 114
].copy()

# Map original symbols to AAMI labels.
processed_114["AAMI_Label"] = (
    processed_114["Label"].map(AAMI_MAP)
)

# Exclude symbols outside the five AAMI beat classes.
processed_114 = processed_114[
    processed_114["AAMI_Label"].notna()
].copy()

processed_114 = processed_114.reset_index(drop=True)
rebuilt_114 = rebuilt_114.reset_index(drop=True)


print("PROCESSED BEATS:", len(processed_114))
print("REBUILT BEATS:", len(rebuilt_114))


# ------------------------------------------------------------
# 2. Compare AAMI class counts
# ------------------------------------------------------------

processed_counts = (
    processed_114["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
)

rebuilt_counts = (
    rebuilt_114["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
)

class_comparison = pd.DataFrame({
    "Existing_Processed": processed_counts,
    "Rebuilt_From_Original": rebuilt_counts
})

class_comparison["Match"] = (
    class_comparison["Existing_Processed"]
    == class_comparison["Rebuilt_From_Original"]
)

print("\nCLASS COUNT COMPARISON")
display(class_comparison)


# ------------------------------------------------------------
# 3. Function to create exact waveform fingerprints
# ------------------------------------------------------------

def create_waveform_counter(df, swap_leads=False):

    result = Counter()

    for row in df[
        ["AAMI_Label"] + WAVEFORM_COLUMNS
    ].itertuples(index=False, name=None):

        label = str(row[0])

        waveform = np.asarray(
            row[1:],
            dtype=np.float64
        )

        if waveform.size != 402:
            raise ValueError(
                f"Expected 402 waveform values, got {waveform.size}"
            )

        # If requested, exchange the two 201-value groups.
        if swap_leads:

            waveform = np.concatenate([
                waveform[201:],
                waveform[:201]
            ])

        waveform = np.ascontiguousarray(waveform)

        digest = hashlib.blake2b(
            waveform.tobytes(),
            digest_size=16
        ).hexdigest()

        result[(label, digest)] += 1

    return result


# ------------------------------------------------------------
# 4. Generate fingerprints for both datasets
# ------------------------------------------------------------

print("\nComparing original lead arrangement...")

processed_normal = create_waveform_counter(
    processed_114,
    swap_leads=False
)

print("Comparing swapped lead arrangement...")

processed_swapped = create_waveform_counter(
    processed_114,
    swap_leads=True
)

print("Creating reconstructed fingerprints...")

rebuilt_counter = create_waveform_counter(
    rebuilt_114,
    swap_leads=False
)


# ------------------------------------------------------------
# 5. Run both comparisons
# ------------------------------------------------------------

normal_match = (
    processed_normal == rebuilt_counter
)

swapped_match = (
    processed_swapped == rebuilt_counter
)

class_counts_pass = bool(
    class_comparison["Match"].all()
)

beat_counts_match = (
    len(processed_114) == len(rebuilt_114)
)


print("\n" + "=" * 65)
print("ACCOUNT A — RECORD 114 VALIDATION RESULTS")
print("=" * 65)

print(
    "Normal arrangement exact match:",
    normal_match
)

print(
    "Swapped arrangement exact match:",
    swapped_match
)

print(
    "AAMI class counts match:",
    class_counts_pass
)

print(
    "Beat counts match:",
    beat_counts_match
)


# ------------------------------------------------------------
# 6. Final interpretation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL INTERPRETATION")
print("=" * 65)

if (
    normal_match
    and class_counts_pass
    and beat_counts_match
):

    print(
        "RESULT: Account A record 114 has the CORRECT "
        "lead arrangement."
    )

elif (
    swapped_match
    and class_counts_pass
    and beat_counts_match
):

    print(
        "RESULT: Account A has the SAME LEAD REVERSAL "
        "as Account B."
    )

else:

    print(
        "RESULT: NEITHER ARRANGEMENT MATCHES COMPLETELY. "
        "Further investigation is required."
    )


print("\nNo dataset files were changed.")
print("No models were trained.")
print("No M0 splits were modified.")

print("\nCELL 5 COMPLETE.")

PROCESSED BEATS: 1879
REBUILT BEATS: 1879

CLASS COUNT COMPARISON


,Existing_Processed,Rebuilt_From_Original,Match
AAMI_Label,,,
N,1820,1820,True
S,12,12,True
V,43,43,True
F,4,4,True
Q,0,0,True



Comparing original lead arrangement...
Comparing swapped lead arrangement...
Creating reconstructed fingerprints...

ACCOUNT A — RECORD 114 VALIDATION RESULTS
Normal arrangement exact match: False
Swapped arrangement exact match: True
AAMI class counts match: True
Beat counts match: True

FINAL INTERPRETATION
RESULT: Account A has the SAME LEAD REVERSAL as Account B.

No dataset files were changed.
No models were trained.
No M0 splits were modified.

CELL 5 COMPLETE.
